In [23]:
from pathlib import Path
import pandas as pd
import numpy as np


# ---------------------------
# CLEAN + LABEL
# ---------------------------
def clean_and_label(
    df: pd.DataFrame,
    flow_ratio_threshold=0.2,
    min_expected_flow=10,
    consecutive_intervals=3
) -> pd.DataFrame:
    """
    Cleans dataframe and computes closure flags using
    relative flow compared to expected hourly baseline.
    """

    stationIds = [
        3054021, 319677, 319673, 319675, 319680, 319674,
        3411021, 3411024, 3023124, 3023121, 319416, 318690,
        317786, 317791, 317789, 317787, 317788, 317797,
        317798, 3412081, 3412064, 3412061, 3047112, 3047111,
        3047113, 3047108, 3047101, 3412054, 3047097, 3047094,
        3047098, 3047084, 3047085, 3047081, 3047073, 3047072,
        3047075, 3047131, 3047042, 3047043, 314000, 316261,
        316249, 3412041, 316214, 316213, 3038021
    ]

    # Filter stations
    df = df[df["Station"].isin(stationIds)].copy()

    # Timestamp
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    df = df.sort_values(["Station", "Timestamp"])

    # Add hour
    df["hour"] = df["Timestamp"].dt.hour

    # ------------------------------------
    # EXPECTED FLOW BASELINE (median)
    # ------------------------------------
    expected_flow = (
        df.groupby(["Station", "hour"])["Total Flow"]
          .median()
          .rename("expected_flow")
    )

    df = df.merge(
        expected_flow,
        on=["Station", "hour"],
        how="left"
    )

    # Prevent unstable ratios from tiny baseline
    df["expected_flow"] = df["expected_flow"].clip(lower=min_expected_flow)

    # ------------------------------------
    # RELATIVE FLOW RATIO
    # ------------------------------------
    df["flow_ratio"] = df["Total Flow"] / df["expected_flow"]

    # Relative low flow
    df["low_flow_relative"] = df["flow_ratio"] < flow_ratio_threshold

    # ------------------------------------
    # Consecutive interval requirement
    # ------------------------------------
    df["closure_flag"] = (
        df.groupby("Station")["low_flow_relative"]
          .transform(
              lambda s: s.rolling(consecutive_intervals,
                                  min_periods=consecutive_intervals)
                        .sum() == consecutive_intervals
          )
    )

    return df


# ---------------------------
# EVENT EXTRACTION
# ---------------------------
def extract_closure_events(df: pd.DataFrame) -> pd.DataFrame:

    df = df.sort_values(["Station", "Timestamp"]).copy()

    df["prev_flag"] = df.groupby("Station")["closure_flag"].shift(fill_value=False)
    df["start_event"] = (~df["prev_flag"]) & (df["closure_flag"])
    df["event_id"] = df.groupby("Station")["start_event"].cumsum()

    closure_rows = df[df["closure_flag"]].copy()

    if closure_rows.empty:
        return pd.DataFrame()

    station_counts = closure_rows.groupby("Timestamp")["Station"].nunique()
    multi_station_times = station_counts >= 2

    closure_rows["multi_station_time"] = (
        closure_rows["Timestamp"].map(multi_station_times).fillna(False)
    )

    events = (
        closure_rows
        .groupby(["Station", "event_id"])
        .agg(
            start_time=("Timestamp", "min"),
            end_time=("Timestamp", "max"),
            n_intervals=("Timestamp", "count"),
            mean_flow=("Total Flow", "mean"),
            mean_speed=("Average Speed", "mean"),
            max_occupancy=("Avg Occupancy", "max"),
            multi_station_event=("multi_station_time", "any")
        )
        .reset_index()
    )

    events["duration_minutes"] = (
        (events["end_time"] - events["start_time"])
        .dt.total_seconds() / 60
        + 5
    )

    events["closure_date"] = events["start_time"].dt.date

    return events


# ---------------------------
# SINGLE FILE PROCESSOR
# ---------------------------
def process_single_file(file_path, save_csv=False):

    columnNames = [
        "Timestamp", "Station", "District#", "Freeway#",
        "Direction of Travel", "Lane Type", "Station Length",
        "Samples", "% Observed", "Total Flow",
        "Avg Occupancy", "Average Speed"
    ]

    unnamedColumns = ["unnamed" + str(i) for i in np.arange(40)]

    df = pd.read_csv(
        file_path,
        compression="gzip",
        header=None,
        names=columnNames + unnamedColumns
    )

    df = df.iloc[:, 0:12]
    df = df[(df["Freeway#"] == 80) & (df["Direction of Travel"] == "E")]
    df = df.dropna()

    df = clean_and_label(df)
    events = extract_closure_events(df)

    if save_csv and not events.empty:
        output_name = Path(file_path).stem.replace(".txt", "") + "_closures.csv"
        events.to_csv(output_name, index=False)
        print(f"Saved to {output_name}")

    return events

def single_file(file_path, save_csv=False):

    columnNames = [
        "Timestamp", "Station", "District#", "Freeway#",
        "Direction of Travel", "Lane Type", "Station Length",
        "Samples", "% Observed", "Total Flow",
        "Avg Occupancy", "Average Speed"
    ]

    unnamedColumns = ["unnamed" + str(i) for i in np.arange(40)]

    df = pd.read_csv(
        file_path,
        compression="gzip",
        header=None,
        names=columnNames + unnamedColumns
    )

    df = df.iloc[:, 0:12]
    df = df[(df["Freeway#"] == 80) & (df["Direction of Travel"] == "E")]
    df = df.dropna()
    return df

In [24]:
file_path = "/Users/hudson/Desktop/road_project/data/unextracted_data/2026/d03_text_station_5min_2026_02_18.txt.gz"
closures = process_single_file(file_path)


In [25]:
closures

,Station,event_id,start_time,end_time,n_intervals,mean_flow,mean_speed,max_occupancy,multi_station_event,duration_minutes,closure_date
0,319673,1,2026-02-18 00:10:00,2026-02-18 00:25:00,4,0.500000,69.375000,0.0013,False,20.0,2026-02-18
1,319673,2,2026-02-18 00:45:00,2026-02-18 01:05:00,5,0.200000,66.440000,0.0077,False,25.0,2026-02-18
2,319673,3,2026-02-18 01:25:00,2026-02-18 02:15:00,11,0.181818,61.445455,0.0006,False,55.0,2026-02-18
3,319673,4,2026-02-18 02:45:00,2026-02-18 03:20:00,8,0.125000,55.762500,0.0008,False,40.0,2026-02-18
4,319673,5,2026-02-18 03:40:00,2026-02-18 04:20:00,9,0.333333,63.611111,0.0006,False,45.0,2026-02-18
5,319673,6,2026-02-18 04:40:00,2026-02-18 04:45:00,2,0.500000,67.900000,0.0005,False,10.0,2026-02-18
6,319673,7,2026-02-18 05:05:00,2026-02-18 05:15:00,3,0.333333,68.233333,0.0008,False,15.0,2026-02-18
7,319673,8,2026-02-18 05:35:00,2026-02-18 05:55:00,5,0.400000,68.900000,0.0008,False,25.0,2026-02-18
8,319673,9,2026-02-18 06:25:00,2026-02-18 06:55:00,7,0.571429,68.871429,0.0006,False,35.0,2026-02-18
9,319673,10,2026-02-18 07:30:00,2026-02-18 07:50:00,5,0.600000,67.660000,0.0008,False,25.0,2026-02-18
